# Advanced Time Series Forecasting, Optimization and Explainability

## Engineering of Intelligent Models - Tutorial #3 (Final Delivery)

### Original Version Full Notebook: "Advanced Topics in Deep Learning - Final Project"

## 1. Introduction and Project Goals

This project builds upon the Time Series Modelling mini-project to develop a systematic, optimized, and well-analysed forecasting system for air temperature prediction using the Jena Climate dataset. While the mini-project focused on exploratory modelling and understanding different design choices, this Final Project elevates performance to a primary objective, achieved through systematic optimization and supported by rigorous analysis.

The core forecasting task is a **multivariate-input, univariate-output, multi-step prediction** problem: given a window of historical meteorological measurements (temperature, pressure, humidity, wind speed, maximum wind speed, and wind direction), the system predicts the air temperature for the next 24 hours.

Our pipeline integrates the following advanced techniques:

- **Baseline Deep Learning Model (GRU):** A configurable multi-layer GRU architecture serves as the reference model against which all improvements are measured, incorporating regularization, gradient clipping, and modern training strategies (AdamW optimizer, Huber loss).

- **Evolutionary Optimization:** A genetic algorithm (GA) automatically searches over an end-to-end pipeline configuration space — jointly optimizing model hyperparameters, architectural choices, training settings, preprocessing strategy, regularization, and windowing parameters. With a population of 20 individuals over 15 generations (300 total evaluations), the EA discovers configurations that outperform the hand-tuned baseline while using fewer parameters.

- **Synthetic Data Generation (TimeGAN):** A Time-series Generative Adversarial Network generates realistic synthetic weather sequences from the training set, demonstrating the feasibility of data augmentation without information leakage.

- **Explainable AI (XAI):** Both global (permutation importance) and local (gradient saliency) explanation methods are applied. XAI insights are further used to guide **feature pruning**, removing redundant features to produce a simpler model with improved performance and robustness.

- **Efficiency and Resource Analysis:** Training time, inference latency, parameter counts, and memory usage (RAM and GPU) are profiled across model configurations, enabling an informed analysis of accuracy–efficiency trade-offs.

The remainder of this report is organized as follows: Section 2 describes the experimental setup and reproducibility measures; Section 3 presents the dataset and problem formulation; Sections 4–6 cover data preparation, feature engineering, and the windowing pipeline; Section 7 establishes the baseline models; Section 8 details the TimeGAN synthetic data generation; Section 9 presents the evolutionary optimization process; Section 10 covers final model retraining, selection, and robustness validation; Sections 11–12 present the explainability and efficiency analyses; and Sections 13–14 provide the comparative discussion and conclusion.

### Data & Configuration

Two DVC-tracked datasets are available for experimentation, both selected via the **data** dropdown in the Configuration Composer:

- **jena_full_dataset** — the full Jena Climate dataset (2009-2016, ~8 years at 10-minute resolution). Use this for final training runs on the complete history.
- **jena_2012_dataset** — a one-year slice containing only 2012, derived from the full file via `src/data/filter_year.py`. Use this for fast iteration while exploring model architectures and hyperparameters.

All training and model hyperparameters are exposed as Configuration Composer overrides so you can tweak them per run without editing code. Every run records its full Hydra configuration bundle to MLflow for lineage and reproducibility.

## 2. Environment, Reproducibility and Experimental Setup

*Methodology — This section and Sections 3–6 describe the data preparation and experimental methodology used throughout the project.*

This section establishes the computational environment, library dependencies, random seed configuration, and project structure. Reproducibility is ensured through fixed seeds, explicit dependency management, and modular code organization.

### 2.1 Libraries Import

In [1]:
import os
import gc
import sys
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import optuna
import psutil

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

### GPU Test

We verify that TensorFlow detects the available GPU and configure memory growth to prevent out-of-memory errors during the evolutionary search, which trains many models sequentially.

In [2]:
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

### 2.2 Utils Configuration Test

We set a global random seed (`SEED = 42`) applied consistently to Python, NumPy, and TensorFlow via `set_global_seed()` to ensure reproducibility. GPU memory growth is enabled via `enable_gpu_memory_growth()`, and device information is printed to confirm the hardware configuration.

In [3]:
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

Set up the project root path and import custom utility functions.

In [4]:
from src.utils.env import set_global_seed, enable_gpu_memory_growth, get_device_info

SEED = int(cfg.seed)
set_global_seed(SEED)
enable_gpu_memory_growth()

print(get_device_info())

We import our custom utility functions and verify the environment setup (seed, GPU memory growth, device info).

## 3. Dataset and Problem Definition

The Jena Climate dataset provides the empirical basis for all experiments conducted in this project. This section presents the dataset, the initial data quality checks, the temporal resampling procedure to hourly resolution, and the formal definition of the forecasting problem, including the target variable and the selected input variables.

### 3.1 Load Data

The Jena Climate dataset is loaded from a local CSV file. The dataset contains meteorological measurements collected at a weather station in Jena, Germany, between January 2009 and December 2016, with an original sampling frequency of 10 minutes across 15 variables.

In [5]:
from src.data.ingestion import ingest

PROJECT_ROOT = Path.cwd()
DATASET_PATH = str(PROJECT_ROOT / cfg.data.file)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)

Load the dataset CSV and preview the first rows.

**Table 1** — First rows of the raw Jena Climate dataset (10-minute sampling, 15 variables).

In [6]:
df, ingest_summary = ingest(DATASET_PATH)

print("Ingestion summary:")
for k, v in ingest_summary.items():
    print(f"  {k}: {v}")

df.head()

Load the CSV file and preview the first rows to confirm the data was read correctly.

### 3.2 Initial Dataset Inspection

Upon loading, we inspect the dataset shape, column names, data types, and general structure. This provides a first look at the raw data before any transformations are applied.

In [7]:
print("Dataset path:", DATASET_PATH)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nInfo:")
df.info()

After loading the raw dataset, we perform quality checks to ensure data integrity before any transformations. Specifically, we verify the presence of duplicated rows and inspect all columns for missing values. Any duplicated observations are removed to avoid biasing the resampling step that follows, duplicate timestamps can arise from logging artifacts and must be eliminated to ensure a consistent temporal index.

In [8]:
print("Duplicated rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isna().sum())

In [9]:
# Deduplication already handled by ingest()
print("Shape after ingestion:", df.shape)
print("Duplicated rows:", df.duplicated().sum())

Remove duplicated rows and verify the cleanup.

### 3.3 Datetime Parsing and Temporal Ordering

The `Date Time` column is parsed into proper datetime objects and the dataframe is sorted chronologically. We verify the date range and inspect the time delta distribution to detect any irregularities in the 10-minute sampling frequency before resampling.

**Table 2** — Dataset date range and first rows after datetime parsing and temporal ordering.

In [10]:
# Datetime parsing already handled by ingest()
print(df["Date Time"].min(), "->", df["Date Time"].max())
df[["Date Time"]].head()

Check the time delta distribution to identify sampling irregularities.

In [11]:
df["Date Time"].diff().value_counts().head(10)

Inspect the distribution of time deltas between consecutive observations to detect any sampling irregularities.

### 3.4 Hourly Resampling

After removing duplicated rows, the cleaned dataset contained 420,224 observations at approximately 10-minute resolution. To align the forecasting problem with hourly dynamics and reduce computational cost, the series was resampled to **1-hour intervals** using `resample("1h").mean()`, which aggregates all observations within each hourly bin through arithmetic averaging.

This operation reduced the dataset from **420,224** rows to **70,129** rows, approximately a sixfold reduction, while preserving the dominant meteorological patterns relevant for day-ahead forecasting. Mean aggregation was preferred over alternatives such as first/last value selection or median aggregation because it preserves the central tendency of each hourly period while smoothing sub-hourly noise that is less relevant for the 24-hour prediction horizon.

In [12]:
from src.data.preprocessing import resample_hourly, select_features, temporal_split

df_hourly, n_nan_dropped = resample_hourly(df)

print("Original shape:", df.shape)
print("Hourly shape:", df_hourly.shape)
print("NaN rows dropped:", n_nan_dropped)
print(df_hourly.head())

### 3.5 Post-Resampling Quality Check

After hourly resampling, the dataset was re-validated to ensure that the transformation did not introduce structural artifacts. In particular, we verified that no duplicated hourly timestamps were created, that the time difference between consecutive observations was predominantly **1 hour**, and that any missing values produced by incomplete hourly bins were explicitly identified.

The quality check showed that the resampled dataset contained **88 missing values per meteorological variable**, while the `Date Time` column remained complete. No duplicated timestamps were found after resampling, and the time-step distribution confirmed that the series was overwhelmingly regular at **1-hour intervals**. This validation step is essential because the supervised sliding-window procedure used later assumes a temporally ordered and nearly regular time series.

In [13]:
print("Missing values after hourly resampling:")
print(df_hourly.isna().sum())

print("\nDuplicated timestamps after resampling:", df_hourly["Date Time"].duplicated().sum())

print("\nTime step distribution after resampling:")
print(df_hourly["Date Time"].diff().value_counts().head(10))

### 3.6 Handling Missing Values After Resampling

All rows containing NaN values after hourly resampling were removed. Because the original dataset has near-complete 10-minute coverage across the 2009–2016 period, the number of affected rows was very small relative to the full hourly dataset. After dropping these rows, the dataset size became **70,041 observations**, and all remaining missing values were eliminated.

Dropping these observations was preferred over imputation because the affected hourly bins were sparse and associated with incomplete aggregation windows or isolated temporal irregularities. In such cases, interpolation could introduce artificial patterns into the series and potentially bias the forecasting models. Although the cleaned series still contains a very small number of larger temporal gaps, the dataset remains overwhelmingly regular at hourly frequency and is suitable for the supervised windowing procedure used in the next stages.

In [14]:
# NaN handling already done by resample_hourly()
print("Shape after resampling:", df_hourly.shape)
print("Missing values:", df_hourly.isna().sum().sum())

### 3.7 Forecasting Task and Variable Selection

Following the project specification, we retain 6 meteorological variables plus the datetime reference: `T (degC)` (air temperature, the target), `p (mbar)` (atmospheric pressure), `rh (%)` (relative humidity), `wv (m/s)` (wind speed), `max. wv (m/s)` (maximum wind speed), and `wd (deg)` (wind direction). All other variables from the original 15-column dataset are excluded.

This defines a **multivariate-input, univariate-output, multi-step** forecasting problem: the model receives a multi-day window of multi-sensor weather data and must predict the air temperature trajectory for the next 24 hours. The multivariate input ensures the model can leverage cross-variable dependencies — for instance, the relationship between falling pressure and subsequent temperature changes driven by weather fronts.

**Table 3** — Selected variables for the forecasting task (6 meteorological features + datetime).

In [15]:
# Feature selection from Hydra config (cfg injected by noted)
selected_columns = list(cfg.data.features)
if "Date Time" not in selected_columns:
    selected_columns = ["Date Time"] + selected_columns

df_model = select_features(df_hourly, selected_columns)

print("Selected columns:")
print(df_model.columns.tolist())
print("\nShape:", df_model.shape)
df_model.head()

## 4. Data Initial Preparation

Before applying feature engineering, the core modelling dataframe is established by defining the target variable and confirming the structure and integrity of the cleaned dataset. This section bridges the data cleaning stage presented in Section 3 with the feature engineering pipeline developed in Section 5, ensuring a clear and consistent transition from raw observations to model-ready inputs.

### 4.1 Target Definition and Base Modeling DataFrame

The target variable is defined as `T (degC)` — air temperature in degrees Celsius. This is the only quantity the model must forecast in the output window; all other variables serve as exogenous inputs.

The `Date Time` column serves as the temporal reference for feature engineering (extracting hour-of-day and day-of-year), but is not included as a direct model input since neural networks cannot interpret raw datetime objects. Instead, its temporal information is encoded as cyclic features in Section 5.

The remaining 5 meteorological variables — atmospheric pressure, relative humidity, wind speed, maximum wind speed, and wind direction — form the covariate set that provides the atmospheric context for the temperature forecast.

In [16]:
TARGET_COL = cfg.data.target
TIME_COL = "Date Time"

feature_cols = [col for col in df_model.columns if col not in [TIME_COL]]
input_feature_cols = [col for col in feature_cols]

print("Target:", TARGET_COL)
print("Time column:", TIME_COL)
print("Input features:", input_feature_cols)
print("Number of input features:", len(input_feature_cols))

### 4.2 Base Forecasting Data Overview

We inspect the prepared dataframe to confirm shape, date range, and column integrity before proceeding to feature engineering. The dataset at this stage contains approximately 70,000 hourly observations spanning from January 2009 to January 2017, with 7 columns (datetime + 6 meteorological variables). This verification step ensures that no data was inadvertently lost during the resampling and cleaning pipeline of Section 3, and establishes the baseline from which all derived features will be computed.

In [17]:
print(df_model.head())
print("\nShape:", df_model.shape)
print("\nDate range:", df_model[TIME_COL].min(), "->", df_model[TIME_COL].max())

## 5. Feature Engineering

Two families of derived features are introduced to enrich the input representation: **cyclical temporal encodings** and **wind-derived features**. The goal is to provide the neural network with representations that respect the physical structure of the data, cyclic quantities are encoded cyclically, and vector quantities are decomposed into Cartesian components. All feature engineering is implemented in `src.features.engineering` for reproducibility.

In [18]:
from src.features.engineering import (
    add_time_features,
    add_wind_features,
    get_final_feature_columns,
)
from src.features.windowing import make_windows
from src.models.gru import build_gru_model

### 5.1 Cyclical Time Features

Hour-of-day and day-of-year carry strong periodic signals (diurnal and seasonal cycles). Encoding them as raw integers would introduce artificial discontinuities (e.g., hour 23 far from hour 0). We apply a sine–cosine encoding that maps each cyclic variable to a point on the unit circle, ensuring smooth transitions at boundaries:
- `hour_sin`, `hour_cos` — diurnal cycle (period = 24h)
- `doy_sin`, `doy_cos` — seasonal cycle (period = 365.25 days)

**Table 4** — Cyclical temporal features derived from hour-of-day and day-of-year.

In [19]:
df_feat = add_time_features(df_model, time_col=TIME_COL)

df_feat[[
    TIME_COL, "hour", "dayofyear",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos"
]].head()

### 5.2 Wind-Derived Features

Wind direction in degrees is inherently circular and difficult for neural networks to interpret directly. We derive six features:
- `wd_sin`, `wd_cos` — cyclic encoding of wind direction
- `wx`, `wy` — Cartesian decomposition of wind vector (speed × cos/sin of direction), replacing the polar representation with a form neural networks process more naturally
- `wind_gap` — difference between max and sustained wind speed (gust intensity)
- `gust_ratio` — ratio of max to sustained wind speed (relative gust strength, with ε = 10⁻⁶ for numerical stability)

**Table 5** — Wind-derived features: cyclic encoding, Cartesian components, gust metrics.

In [20]:
df_feat = add_wind_features(df_feat)

df_feat[[
    TIME_COL, "wv (m/s)", "max. wv (m/s)", "wd (deg)",
    "wd_sin", "wd_cos", "wx", "wy", "wind_gap", "gust_ratio"
]].head()

### 5.3 Final Feature Set for Modeling

The complete input feature vector contains **16 dimensions**: 6 original meteorological variables + 4 cyclical temporal encodings + 6 wind-derived features. This feature set is used consistently across all experiments, baseline, evolutionary optimization, and synthetic data generation. The XAI analysis in Section 11 later evaluates which of these 16 features contribute most, leading to a pruned 11-feature variant.

In [21]:
final_feature_cols = get_final_feature_columns()

print("Number of modeling features:", len(final_feature_cols))
print(final_feature_cols)

## 6. Split, Scaling and Windowing

This section completes the data preparation pipeline by defining the temporal train/validation/test split, applying feature scaling without data leakage, and transforming the time series into supervised learning windows. These steps produce the final input–output tensors used by all forecasting models throughout the project.

### 6.1 Temporal Train/Validation/Test Split

The dataset is split chronologically (70% / 15% / 15%) with **no shuffling**, the split respects temporal order to prevent data leakage. The training set covers the earliest years, followed by validation, with the most recent data reserved for testing. This mirrors a realistic deployment scenario where the model is trained on historical data and evaluated on future observations.

In [22]:
df_train, df_val, df_test = temporal_split(
    df_feat,
    train_ratio=cfg.data.split.train,
    val_ratio=cfg.data.split.val,
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

### 6.2 Feature Scaling

All input features are normalized using **StandardScaler** (zero mean, unit variance) as the baseline strategy. The scaler is fit exclusively on the training set and applied via `transform` to validation and test sets, preventing information leakage. The evolutionary optimization also searches over the scaler type (standard, robust, minmax), allowing the GA to discover whether a different normalization strategy benefits specific model configurations.

In [23]:
from src.features.scaling import get_scaler

Apply the selected scaler to all splits (fit on train, transform on val/test).

In [24]:
SCALER_NAME = cfg.scaler.name
scaler = get_scaler(SCALER_NAME)

X_train_df = df_train[final_feature_cols].copy()
X_val_df = df_val[final_feature_cols].copy()
X_test_df = df_test[final_feature_cols].copy()

X_train_scaled = scaler.fit_transform(X_train_df)
X_val_scaled = scaler.transform(X_val_df)
X_test_scaled = scaler.transform(X_test_df)

print("Scaler:", SCALER_NAME)
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Apply the scaler to the training, validation, and test feature matrices. The scaler is fit only on the training data.

### 6.3 Target Index for Forecasting

We identify the position of `T (degC)` within the 16-feature vector. This index is used during windowing to extract only the temperature column for the output sequence **y**, while keeping all 16 features in the input tensor **X**. This separation makes the problem multivariate-input but univariate-output.

In [25]:
target_idx = final_feature_cols.index(TARGET_COL)

print("Target column:", TARGET_COL)
print("Target index:", target_idx)

### 6.4 Supervised Windowing

The scaled multivariate time series is converted into supervised learning samples through a sliding-window procedure. For each sample, the input tensor **X** contains a sequence of consecutive hourly observations across all modeling features, while the output vector **y** contains only the future air temperature values to be predicted.

Under the default configuration, the input has shape **`(LOOKBACK, 16)`** and the output has shape **`(HORIZON,)`**, corresponding to a multivariate-input, univariate-output, multi-step forecasting setup. In the base configuration, `LOOKBACK = 120` hours (5 days) and `HORIZON = 24` hours (1 day ahead). In the evolutionary optimization stage presented in Section 9, alternative lookback lengths are also explored as part of the search space, so the windowing strategy is not treated as fixed throughout the project.

In [26]:
from src.features.windowing import make_windows

Create supervised windows for all splits using the defined lookback and horizon.

In [27]:
LOOKBACK = cfg.data.lookback
HORIZON = cfg.data.horizon

X_train, y_train = make_windows(X_train_scaled, target_idx, LOOKBACK, HORIZON)
X_val, y_val = make_windows(X_val_scaled, target_idx, LOOKBACK, HORIZON)
X_test, y_test = make_windows(X_test_scaled, target_idx, LOOKBACK, HORIZON)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

Create supervised windows for train, validation, and test splits using the defined lookback (120h) and horizon (24h).

### 6.5 Windowed Data Sanity Check

A final sanity check is performed on the windowed data to confirm that the supervised transformation produced tensors with the expected structure and dimensionality. In particular, the input windows follow the shape **`(N, LOOKBACK, 16)`**, while the target arrays follow the shape **`(N, HORIZON)`**, corresponding to multivariate input sequences and 24-step univariate temperature targets.

The inspection confirms that each input sample contains **120 hourly observations across 16 features**, and that each target sample contains **24 future temperature values**. This validation step is important because it helps detect off-by-one errors or alignment mistakes in the windowing process that could silently compromise model training and evaluation.

In [28]:
print("Input shape:", X_train.shape[1:])
print("Forecast horizon:", y_train.shape[1])
print("Number of input features:", X_train.shape[2])

print("\nExample input window shape:", X_train[0].shape)
print("Example target shape:", y_train[0].shape)
print("First 5 target values (scaled):", y_train[0][:5])

## 7. Baseline Models

Sections 7 to 9 present the core experimental pipeline of the project, covering baseline establishment, synthetic data generation, and evolutionary optimization.

Two baseline models are defined as reference points for all subsequent analyses. The **persistence baseline** provides a minimal no-skill benchmark that any learned forecasting model should outperform. The **GRU baseline** represents a stronger deep learning reference model, trained under a modern and carefully controlled setup, and serves as the main benchmark against which the optimized evolutionary solutions are evaluated.

### 7.1 Persistence Baseline

The persistence baseline corresponds to the simplest possible forecasting strategy: the last observed temperature value is repeated across the entire 24-hour forecast horizon. This naive approach provides a minimal reference level for performance, since any learned forecasting model should be able to outperform it.

The resulting prediction tensor has shape **`(10364, 24)`**, matching the multi-step forecasting target, and each prediction consists of a constant repetition of the final observed temperature from the corresponding input window.

In [29]:
def persistence_forecast(X):
    # Repeats the last observed target value across the full forecast horizon
    last_temp = X[:, -1, target_idx]
    return np.repeat(last_temp[:, None], HORIZON, axis=1)

y_pred_persistence = persistence_forecast(X_test)

print("Persistence prediction shape:", y_pred_persistence.shape)
print("First prediction:", y_pred_persistence[0][:5])

Compute scaled metrics for the persistence forecast on the test set.

In [30]:
persistence_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_persistence.flatten())
persistence_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_persistence.flatten()))

print("Persistence MAE (scaled):", persistence_mae_scaled)
print("Persistence RMSE (scaled):", persistence_rmse_scaled)

Compute persistence baseline metrics on the test set.

In [31]:
def build_gru_baseline(input_shape, horizon):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

gru_baseline = build_gru_baseline(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    horizon=HORIZON
)

gru_baseline.summary()

Before building the official configurable baseline, we construct a simple 1-layer GRU (64 units) as a quick sanity check. This verifies that the full pipeline — from windowed data through model training to test evaluation — works correctly end-to-end. The simple model's results are not used for comparison; they serve only as a development checkpoint.

In [32]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_baseline = gru_baseline.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Train the simple GRU with early stopping on validation loss.

In [33]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:5])

Generate and inspect test set predictions from the simple GRU.

In [34]:
gru_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_gru.flatten())
gru_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_gru.flatten()))

print("GRU MAE (scaled):", gru_mae_scaled)
print("GRU RMSE (scaled):", gru_rmse_scaled)

Evaluate the simple GRU on scaled metrics to compare against persistence.

### 7.2 GRU Baseline

The official GRU baseline was defined using the best-performing configuration identified in the previous Time Series Modelling mini-project. This ensures methodological continuity and provides a strong, previously validated reference model for the current project.

The model is implemented through `build_gru_model()` from `src.models.gru` and consists of a configurable 2-layer GRU architecture with **96** and **64** hidden units, respectively. The baseline also includes an intermediate dense layer with **256** units, **AdamW** optimization, **Huber loss** with \( \delta = 1.0 \), **L2 regularization**, and **gradient clipping**. Training is performed with **early stopping** (`patience=6`) and **learning rate reduction on plateau** (`patience=3`, `factor=0.5`), providing a stable and competitive deep learning benchmark against which the evolutionary optimization will be evaluated.

In [35]:
from src.models.gru import build_gru_model

Define the official baseline configuration and build the model.

In [36]:
from src.training.pipeline import build_model_from_cfg, train_pipeline

# Load model config from Hydra (cfg.model contains the active model config)
BASELINE_CFG = {
    "n_layers": cfg.model.n_layers,
    "units1": cfg.model.units1,
    "units2": cfg.model.units2,
    "units3": cfg.model.units3,
    "dropout": cfg.model.dropout,
    "l2": cfg.model.l2,
    "dense_units": cfg.model.dense_units,
    "dense_activation": cfg.model.dense_activation,
    "learning_rate": cfg.training.learning_rate,
    "clipnorm": cfg.training.clipnorm,
    "optimizer_name": cfg.model.optimizer_name,
    "weight_decay": cfg.model.weight_decay,
    "loss_name": cfg.model.loss_name,
    "gaussian_noise_std": cfg.model.gaussian_noise_std,
    "batch_size": cfg.training.batch_size,
    "scaler_name": cfg.scaler.name,
}

gru_baseline = build_model_from_cfg(BASELINE_CFG, LOOKBACK, X_train.shape[2], HORIZON)
gru_baseline.summary()

In [37]:
from src.models.train_eval import (
    train_model,
    evaluate_scaled_forecasts,
    inverse_scale_target,
    evaluate_original_scale_forecasts,
)

Import training and evaluation utilities.

In [38]:
# MLflow epoch logging callback (streams metrics to Live Metrics panel)
import mlflow

class _MLflowEpochLogger(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        total = self.params.get("epochs", 0)
        print(f"TOTAL EPOCHS: {total}")
        if mlflow.active_run() and total > 0:
            mlflow.log_metric("total_epochs", total)
    def on_epoch_end(self, epoch, logs=None):
        if logs and mlflow.active_run():
            for key, value in logs.items():
                mlflow.log_metric(key, float(value), step=epoch + 1)

# Inject callback into model.fit via monkey-patch
_orig_fit = gru_baseline.fit
def _fit_with_logging(*args, **kwargs):
    cbs = list(kwargs.get("callbacks", []) or [])
    cbs.append(_MLflowEpochLogger())
    kwargs["callbacks"] = cbs
    return _orig_fit(*args, **kwargs)
gru_baseline.fit = _fit_with_logging

history_baseline = train_model(
    model=gru_baseline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=cfg.training.batch_size,
    epochs=cfg.training.epochs,
    es_cfg=cfg.training.early_stopping,
    lr_cfg=cfg.training.lr_reduction,
    verbose=1
)


The official baseline is trained with the epoch budget configured in `cfg.training.epochs`. Training uses early stopping (with `cfg.training.early_stopping.patience` and `restore_best_weights`) and learning rate reduction on plateau (with `cfg.training.lr_reduction.factor`, `patience` and `min_lr`). All these values can be tweaked live in the Configuration Composer before each run, so you can experiment with different training regimes without editing code.

In [39]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:12])

Generate test predictions from the official baseline and inspect the output.

In [40]:
gru_scaled_metrics = evaluate_scaled_forecasts(y_test, y_pred_gru)

gru_mae_scaled = gru_scaled_metrics["mae_scaled"]
gru_rmse_scaled = gru_scaled_metrics["rmse_scaled"]

print("GRU Baseline MAE (scaled):", gru_mae_scaled)
print("GRU Baseline RMSE (scaled):", gru_rmse_scaled)

Compute scaled MAE and RMSE for the official GRU baseline on the test set.

### 7.3 Baseline Comparison

Table 6 presents a side-by-side comparison of the persistence baseline and the official GRU baseline using **scaled evaluation metrics**, namely MAE and RMSE computed in normalized space. These metrics are useful for comparing models trained under the same scaling regime and for monitoring optimization behaviour during training.

However, scaled errors do not have a direct physical interpretation. For this reason, the comparison is extended in the following subsections by converting both predictions and targets back to the original temperature scale, allowing the results to be interpreted in degrees Celsius.

**Table 6** — Persistence vs. GRU baseline comparison on scaled metrics (test set).

In [41]:
baseline_results_scaled = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
})

baseline_results_scaled

### 7.4 Reverse Scaling

Predictions and ground truth are inverse-transformed from normalized space back to degrees Celsius using the training set's scaler statistics (mean and standard deviation of the temperature column). This allows evaluation in physically meaningful units.

In [42]:
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

y_test_inv = inverse_scale_target(y_test, target_mean, target_std)
y_pred_persistence_inv = inverse_scale_target(y_pred_persistence, target_mean, target_std)
y_pred_gru_inv = inverse_scale_target(y_pred_gru, target_mean, target_std)

print("y_test_inv shape:", y_test_inv.shape)
print("y_pred_gru_inv shape:", y_pred_gru_inv.shape)

### 7.5 Baseline Evaluation in Original Temperature Scale

We compute MAE and RMSE in degrees Celsius on the inverse-scaled predictions. These original-scale metrics are the primary comparison basis throughout the report — an MAE of 1.65°C means the model's 24-hour forecasts are off by an average of 1.65 degrees. This is the metric used as the evolutionary fitness function and the final evaluation criterion.

In [43]:
gru_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_gru_inv)
persistence_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_persistence_inv)

gru_mae = gru_original_metrics["mae"]
gru_rmse = gru_original_metrics["rmse"]

persistence_mae = persistence_original_metrics["mae"]
persistence_rmse = persistence_original_metrics["rmse"]

print("GRU Baseline MAE (°C):", gru_mae)
print("GRU Baseline RMSE (°C):", gru_rmse)

### 7.6 Final Baseline Comparison

Table 7, combines both **scaled** and **original-scale** evaluation metrics for the persistence and GRU baseline models. The official GRU baseline substantially outperforms the naive persistence forecast, reducing the test error from **3.144 °C** to **1.665 °C** in MAE and from **4.254 °C** to **2.209 °C** in RMSE.

These results confirm that the recurrent architecture is able to capture meaningful temporal dependencies in the meteorological series and establish a strong benchmark against which the subsequent optimization stages will be evaluated.

**Table 7** — Final baseline comparison with both scaled and original-scale (°C) metrics.

In [44]:
baseline_results = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
    "MAE_degC": [persistence_mae, gru_mae],
    "RMSE_degC": [persistence_rmse, gru_rmse],
})

baseline_results

### 7.7 Forecast Visualization

Figure 1, presents a visual comparison of a single 24-hour forecast window on the test set, showing the true future temperature trajectory together with the persistence and GRU baseline predictions in degrees Celsius.

The persistence baseline produces a flat forecast, since it simply repeats the last observed temperature across the full horizon. As expected, this strategy fails to capture the strong upward and downward variations visible in the true trajectory. In contrast, the GRU baseline is able to follow the general temporal pattern more closely, correctly anticipating the broad rise in temperature during the first half of the horizon and the subsequent declining trend.

Although the GRU forecast is smoother than the true signal and underestimates the amplitude of some changes, it still represents a substantial improvement over persistence. This qualitative comparison is consistent with the quantitative results reported earlier and confirms that the recurrent model captures meaningful temporal structure in the meteorological series.

In [45]:
sample_idx = 0

plt.figure(figsize=(10, 5))
plt.plot(y_test_inv[sample_idx], label="True", marker="o")
plt.plot(y_pred_persistence_inv[sample_idx], label="Persistence", linestyle="--")
plt.plot(y_pred_gru_inv[sample_idx], label="GRU Baseline", linestyle="--")
plt.title("Example Multi-step Forecast on Test Set")
plt.xlabel("Forecast Step")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.show()

**Figure 1** — 24-hour forecast comparison: ground truth vs. persistence vs. GRU baseline (°C).

## 8. Data Quality Assessment with Evidently

Before proceeding to model registration, we generate an Evidently data quality report
to profile the dataset and establish a baseline for future drift detection.

In [46]:
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataSummaryPreset
from evidently.ui.workspace import RemoteWorkspace

EVIDENTLY_PROJECT_NAME = "Jena Weather"

data_def = DataDefinition(
    numerical_columns=final_feature_cols,
    timestamp="Date Time",
)
ev_dataset = Dataset.from_pandas(df_feat, data_definition=data_def)

quality_report = Report([DataSummaryPreset()], tags=["data-quality", "jena-weather", "run-1"])
quality_snapshot = quality_report.run(ev_dataset)

ws = RemoteWorkspace("http://noted-evidently:8000")
# Get or create the Evidently project by name
matches = [p for p in ws.list_projects() if p.name == EVIDENTLY_PROJECT_NAME]
if matches:
    project = matches[0]
else:
    project = ws.create_project(EVIDENTLY_PROJECT_NAME)
    print(f"Created Evidently project: {project.id}")
ws.add_run(project.id, quality_snapshot, include_data=False)

print("Data quality report saved to Evidently workspace")

## 9. Model Registration and Auto-Promotion

After evaluation, the trained model is registered in the MLflow Model Registry.
If it outperforms the current champion, it is automatically promoted.

In [47]:
import mlflow
import mlflow.tensorflow
from src.evaluation.promote import register_and_promote

# Only log to MLflow if Run Manager has opened a run for us.
# Run All / Play button leaves mlflow.active_run() as None and the whole
# block below is skipped - by design, only Run Manager produces MLflow runs.
run = mlflow.active_run()
if run is not None:
    mlflow.set_tracking_uri("http://mlflow:5000")
    mlflow.set_experiment("jena_weather")

    # Log parameters from Hydra config
    mlflow.log_params({
        "model_type": cfg.model.type,
        "units1": cfg.model.units1,
        "units2": cfg.model.units2,
        "n_layers": cfg.model.n_layers,
        "dropout": cfg.model.dropout,
        "lookback": cfg.data.lookback,
        "horizon": cfg.data.horizon,
        "scaler": cfg.scaler.name,
        "epochs": cfg.training.epochs,
        "batch_size": cfg.training.batch_size,
    })

    # Log metrics
    mlflow.log_metrics({
        "mae_scaled": gru_scaled_metrics["mae_scaled"],
        "rmse_scaled": gru_scaled_metrics["rmse_scaled"],
        "test_mae_degC": gru_original_metrics["mae"],
        "test_rmse_degC": gru_original_metrics["rmse"],
    })

    # Log Hydra config hash
    mlflow.set_tag("hydra_config_hash", __noted_hydra_hash__)

    # Log model
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X_train, y_pred_gru)
    mlflow.tensorflow.log_model(gru_baseline, name="model", signature=signature)

    print(f"MLflow run ID: {run.info.run_id}")
    print(f"Config hash: {__noted_hydra_hash__}")
else:
    print("No active MLflow run (Run All path). Skipping MLflow logging.")
    print("Use Run Manager to train and register this model.")


In [48]:
# Auto-promote if better than current champion
result = register_and_promote(
    model=gru_baseline,
    model_name="Jena Weather Forecaster",
    run_id=run.info.run_id,
    new_mae=gru_original_metrics["mae"],
)

print(f"Registered: v{result['new_version']}")
print(f"Promoted: {result['promoted']}")
if result["promoted"]:
    imp = result.get("improvement_pct")
    if imp is not None:
        print(f"Improvement: {imp:.1f}%")
    else:
        print("Improvement: N/A (first champion, no baseline)")


## 10. Data Drift Detection

To demonstrate drift detection, we compare the training data distribution
against the test data distribution. In a production scenario, this would
compare training data against new incoming data.

In [98]:
from evidently.presets import DataDriftPreset

# Reference: training data, Current: test data
ref_def = DataDefinition(numerical_columns=final_feature_cols)
ref_dataset = Dataset.from_pandas(df_train[final_feature_cols], data_definition=ref_def)
cur_dataset = Dataset.from_pandas(df_test[final_feature_cols], data_definition=ref_def)

drift_report = Report([DataDriftPreset()],
    tags=["drift", "jena-weather", "train-vs-test"])
drift_snapshot = drift_report.run(cur_dataset, ref_dataset)

ws.add_run(project.id, drift_snapshot, include_data=False)

print("Drift report saved to Evidently workspace")
print("\nDrift results:")
drift_dict = drift_snapshot.dict()
for metric in drift_dict.get("metrics", []):
    name = metric.get("metric_name", "")
    if "Drift" in name:
        print(f"  {name}: {metric.get('value', '')}")

## 11. Summary

This notebook demonstrated the complete MLOps lifecycle:

1. **Data Ingestion** - automated loading, validation, and cleaning
2. **Preprocessing** - hourly resampling, feature engineering (16 features)
3. **Configuration** - all parameters managed via Hydra (model, training, scaler, data)
4. **Data Quality** - Evidently profiling report saved to workspace
5. **Training** - GRU model built from Hydra config, trained with early stopping
6. **Evaluation** - scaled and original-scale metrics (MAE, RMSE)
7. **Registration** - model registered in MLflow Registry with auto-promotion
8. **Drift Detection** - Evidently comparison of training vs test distributions

All configuration is managed by Hydra, all metrics tracked by MLflow,
all data versioned by DVC, all reports stored in Evidently.
The same `src/` modules can be orchestrated by Airflow DAGs for automated execution.